In [1]:
import tensorflow_advanced_segmentation_models as tasm
import tensorflow as tf
from utils import file
import matplotlib.pyplot as plt 

BACKBONE_NAME = "efficientnetb0"
WEIGHTS = "imagenet"
HEIGHT = 256
WIDTH = 256

In [2]:
base_model, layers, layer_names = tasm.create_base_model(name=BACKBONE_NAME, weights=WEIGHTS, height=HEIGHT, width=WIDTH)

In [10]:
model = tasm.FPNet(n_classes=1, base_model=base_model, output_layers=layers, backbone_trainable=False)
model.compile(tf.keras.optimizers.Adam(0.0001), loss=tasm.losses.DiceLoss(), metrics=["accuracy", tasm.metrics.FScore(), tasm.metrics.IOUScore()],)

In [12]:
TRAINING_IMAGE_PATH = 'training/images'
TRAINING_MASK_PATH = 'training/masks'
trainingImagePaths = file.getSortedFilePaths(TRAINING_IMAGE_PATH)
trainingMaskPaths = file.getSortedFilePaths(TRAINING_MASK_PATH)
trainDataset = file.datasetGenerator(trainingImagePaths, trainingMaskPaths)
trainDataset = trainDataset.prefetch(buffer_size=tf.data.AUTOTUNE)
trainDataset

<PrefetchDataset element_spec=(TensorSpec(shape=(None, 256, 256, 3), dtype=tf.uint8, name=None), TensorSpec(shape=(None, 256, 256, 1), dtype=tf.float32, name=None))>

In [13]:
TESTING_IMAGE_PATH = 'testing/images'
TESTING_MASK_PATH = 'testing/masks'
testingImagePaths = file.getSortedFilePaths(TESTING_IMAGE_PATH)
testingMaskPaths = file.getSortedFilePaths(TESTING_MASK_PATH)
testDataset = file.datasetGenerator(testingImagePaths, testingMaskPaths)
testDataset = testDataset.prefetch(buffer_size=tf.data.AUTOTUNE)
testDataset

<PrefetchDataset element_spec=(TensorSpec(shape=(None, 256, 256, 3), dtype=tf.uint8, name=None), TensorSpec(shape=(None, 256, 256, 1), dtype=tf.float32, name=None))>

In [6]:
current_loss = "dice"

class DisplayCallback(tf.keras.callbacks.Callback):
    def __init__(self, dataset, model, epoch_interval=2):
        self.dataset = dataset
        self.epoch_interval = epoch_interval
        self.best_val_score = 0.0
        self.model = model
    
    def display(self, display_list, extra_title=''):
        plt.figure(figsize=(15, 15))
        title = ['Tikroji nuotrauka', 'Tikroji segmentacija', 'Segmentuota nuotrauka']

        if len(display_list) > len(title):
            title.append(extra_title)

        for i in range(len(display_list)):
            plt.subplot(1, len(display_list), i+1)
            plt.title(title[i])
            plt.imshow(display_list[i], cmap='gray')
            plt.axis('off')
        plt.show()
        
    def create_mask(self, pred_mask):
        pred_mask = tf.argmax(pred_mask, axis=-1)
        pred_mask = pred_mask[..., tf.newaxis]
        return pred_mask[0]
    
    def show_predictions(self, dataset, num=2):
        for image, mask in dataset.take(num):
            pred_mask = self.model.predict(image)
            self.display([image[0], mask[0], self.create_mask(pred_mask)])
        
    def on_epoch_end(self, epoch, logs=None):
        # also save if validation error is smallest
        print(logs.keys())
        if 'f1-score' in logs.keys():
            val_score = logs['f1-score']
            if val_score > self.best_val_score:
                self.best_val_score = val_score
                print('New best weights found!')
                self.model.save_weights("./fpnet-weights/" + current_loss + '/best_weights.hdf5')
        else:
            print('Key val_dice_eval does not exist!')
            
        if epoch and epoch % self.epoch_interval == 0:
            self.show_predictions(self.dataset)
            print ('\nSample Prediction after epoch {}\n'.format(epoch+1))

from keras.callbacks import ModelCheckpoint


mcp_save = ModelCheckpoint(
    './fpnet-weights/' +  current_loss + '/weights.{epoch:02d}-{loss:.4f}.hdf5', save_best_only=False, save_weights_only=True, monitor='loss', verbose=1)

In [11]:
history = model.fit(
    trainDataset,
    batch_size=12,
    epochs=50,
    validation_data=testDataset,
    shuffle=True,
    callbacks=[DisplayCallback(trainDataset, model), mcp_save],
)

Epoch 1/50


ValueError: in user code:

    File "c:\Users\ITWORK\miniconda3\lib\site-packages\keras\engine\training.py", line 1021, in train_function  *
        return step_function(self, iterator)
    File "c:\Users\ITWORK\miniconda3\lib\site-packages\keras\engine\training.py", line 1010, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "c:\Users\ITWORK\miniconda3\lib\site-packages\keras\engine\training.py", line 1000, in run_step  **
        outputs = model.train_step(data)
    File "c:\Users\ITWORK\miniconda3\lib\site-packages\keras\engine\training.py", line 859, in train_step
        y_pred = self(x, training=True)
    File "c:\Users\ITWORK\miniconda3\lib\site-packages\keras\utils\traceback_utils.py", line 67, in error_handler
        raise e.with_traceback(filtered_tb) from None

    ValueError: Exception encountered when calling layer "fp_net_2" (type FPNet).
    
    in user code:
    
        File "c:\Users\ITWORK\miniconda3\lib\site-pac

In [ ]:
import pandas as pd
history = model.history.history
history_df = pd.DataFrame(history)
history_df.to_csv('deeplab_focal_history.csv', index=True)

In [ ]:
model.load_weights("./deeplab-weights/" +  current_loss + "/best_weights.hdf5")

scores = model.evaluate(testDataset)
print("Loss: {:.5}".format(scores[0]))
for metric, value in zip(["accuracy", tasm.metrics.FScore(), tasm.metrics.IOUScore()], scores[1:]):
    if metric != "accuracy":
        metric = metric.__name__
    print("mean {}: {:.5}".format(metric, value))